## Current v-LSTM Model:

In [ ]:
# Cell 2: Vanilla LSTM (multi-step multi-input)
# Headings: Processing data, Feature engineering & selection, Training model, Test model
# Expects variables from Cell 1: per_subject_train, per_subject_test, X_tr_all, y_tr_all, cg_scaler

import numpy as np
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, TimeDistributed
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# ---------- Processing data (use aggregated arrays from preproc cell) ----------
X_train = X_tr_all  # shape (N_samples, n_in, n_features)
y_train = y_tr_all  # shape (N_samples, n_out, 1)
batch_size = 64
#epochs = 20
epochs = 2  # reduced for quicker testing
n_in = X_train.shape[1]
n_feats = X_train.shape[2]
n_out = y_train.shape[1]

# ---------- Feature engineering: no further features added; keep shape as is ----------
# ---------- Custom surrogate loss (differentiable approximation of compound metric) ----------
# We use RMSE + alpha * soft-glycemia mismatch (soft classes via logistic around thresholds)
alpha = 0.6
k_soft = 0.08  # slope for soft thresholds (tuneable)

# paper constants
C = 32.9170
w_left = 19.0
w_right = 1.0
T_mgdl = 105.0  # target BG in mg/dL

# slope-cost constants (tuneable)
a = 1.0
b = 1.0

eps = 1e-8

# helper: convert scaled values back to mg/dL using your scaler
min_v = float(cg_scaler.data_min_[0])
max_v = float(cg_scaler.data_max_[0])

def _to_mgdl(tensor_scaled):
    """tensor_scaled: shape (batch, n_out) or (...). Inverse min-max scaling to mg/dL."""
    return tensor_scaled * (max_v - min_v) + min_v

def _zone_cost(bg_mgdl):
    """bg_mgdl: tensor of BG in mg/dL, any shape. Returns zone cost same shape."""
    # log difference squared
    # add eps to log to avoid -inf/nan
    log_bg = tf.math.log(bg_mgdl + eps)
    log_T = tf.math.log(T_mgdl + eps)
    diff2 = tf.square(log_bg - log_T)

    # piecewise multiply by C*w_left or C*w_right
    left_mask = tf.cast(bg_mgdl < T_mgdl, dtype=bg_mgdl.dtype)
    right_mask = 1.0 - left_mask
    cost = C * (w_left * diff2 * left_mask + w_right * diff2 * right_mask)
    return cost

def _slope_cost(delta_bg_mgdl):
    """delta_bg_mgdl: difference in mg/dL. Uses formula in paper but converted to mmol/L inside."""
    # convert mg/dL -> mmol/L by /18.0
    delta_mmol = delta_bg_mgdl / 18.0
    # piecewise: if delta < 0 use 2b*(Δ)^2, else a*(Δ)^2
    neg_mask = tf.cast(delta_mmol < 0.0, dtype=delta_mmol.dtype)
    pos_mask = 1.0 - neg_mask
    cost = (2.0 * b) * tf.square(delta_mmol) * neg_mask + a * tf.square(delta_mmol) * pos_mask
    return cost

def lstm_paper_loss(y_true, y_pred):
    """
    y_true, y_pred: expected shapes (batch, n_out) because you flatten targets before fit.
    Returns scalar loss: mean( weight * squared_error ) + alpha * mean( abs_error )
    where weight = zone_cost(y_true_mgdl) + slope_cost(delta_y_true_mgdl) + 1
    """
    # ensure float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # convert to mg/dL
    y_true_mgdl = _to_mgdl(y_true)
    y_pred_mgdl = _to_mgdl(y_pred)

    # compute slope/delta along horizon axis (axis=1). For first horizon delta=0
    # shape (batch, n_out)
    y_true_shifted = tf.concat([y_true_mgdl[:, :1] * 0.0, y_true_mgdl[:, :-1]], axis=1)
    delta_true = y_true_mgdl - y_true_shifted

    # zone cost and slope cost per element
    zone = _zone_cost(y_true_mgdl)          # shape (batch,n_out)
    slope = _slope_cost(delta_true)        # shape (batch,n_out)
    weights = zone + slope + 1.0           # baseline +1 as in paper

    # weighted MSE (average over all elements)
    se = tf.square(y_pred - y_true)        # in scaled units
    weighted_se = weights * se
    mse_weighted = tf.reduce_mean(weighted_se)

    # L1 term (in scaled units)
    l1 = tf.reduce_mean(tf.abs(y_pred - y_true))

    # Paper returns MSE_weighted + alpha * L1
    loss = mse_weighted + alpha * l1

    return loss



def soft_glycemia_probs(y):
    # y: tensor of glucose values in scaled space -> convert to mg/dL by inverse scaling inside loss
    y = tf.reshape(y, (-1,))  # flatten
    # inverse scale using stored scaler parameters (min/max). We compute mg/dL approximations:
    min_v = float(cg_scaler.data_min_[0]); max_v = float(cg_scaler.data_max_[0])
    y_mgdl = y * (max_v - min_v) + min_v
    # probabilities: hypo prob ~ sigmoid(k*(70 - y)), hyper prob ~ sigmoid(k*(y - 180))
    hypo = tf.sigmoid(k_soft * (70.0 - y_mgdl))
    hyper = tf.sigmoid(k_soft * (y_mgdl - 180.0))
    norm = 1.0 - tf.clip_by_value(hypo + hyper, 0.0, 1.0)
    probs = tf.stack([hypo, norm, hyper], axis=1)
    return probs

def lstm_surrogate_loss(y_true, y_pred):
    # y_true/y_pred shapes (batch, n_out, 1)
    # RMSE term
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    rmse = tf.sqrt(mse + 1e-8)
    # soft glycemia mismatch: compute probs and L1 difference
    pt = soft_glycemia_probs(y_true)
    pp = soft_glycemia_probs(y_pred)
    soft_diff = tf.reduce_mean(tf.abs(pt - pp))
    return rmse + alpha * soft_diff

# ---------- Training model ----------
tf.keras.backend.clear_session()
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(n_in, n_feats)),
    LSTM(32, return_sequences=False),
    Dense(n_out*1),
    # reshape at training-time to (n_out,1) in loss; Keras expects final shape (batch, n_out)
])
# we'll wrap outputs to (batch, n_out, 1) by custom lambda at training time in fit callbacks,
# but simplest is to compile with target flattened to (n_out,) so cast both
model.add(Dense(n_out, activation='linear'))

model.compile(optimizer=Adam(learning_rate=1e-3), loss=lstm_paper_loss)
print(model.summary())

# prepare training targets flattened to (batch, n_out)
y_train_flat = y_train.reshape((y_train.shape[0], n_out))

model.fit(X_train, y_train_flat, epochs=epochs, batch_size=batch_size, verbose=2)

# ---------- Test model: prediction helper ----------
def lstm_predict_on_subject(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0]==0:
        return np.zeros((0, n_out,1))
    X = info['X']
    y_hat_flat = model.predict(X, batch_size=64)
    y_hat = y_hat_flat.reshape((-1, n_out,1))
    return y_hat

# expose model variable for later evaluation
lstm_model = model


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, concatenate, Masking, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import random, os

# ✅ Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ✅ Configuration
DATA_PATH = "../data/OhioT1DM.csv"
SUBJECT_ID = None    # None -> picks the first subject
LOOKBACK_MIN = 60    # past window (minutes)
PRED_HORIZ_MIN = 30  # forecast horizon (minutes)
STEP_MIN = 5
TEST_FRACTION = 0.2
BATCH_SIZE = 16
EPOCHS = 100
LSTM_UNITS = 64
DROPOUT = 0.2

LOOKBACK_STEPS = LOOKBACK_MIN // STEP_MIN
N_PRED_STEPS = PRED_HORIZ_MIN // STEP_MIN

print("Lookback steps:", LOOKBACK_STEPS,
      "| Prediction steps:", N_PRED_STEPS)

# ✅ Load data
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df = df.sort_values(['id', 'date']).reset_index(drop=True)

# ✅ Select subject
if SUBJECT_ID is None:
    SUBJECT_ID = df['id'].unique()[0]
print("Using subject:", SUBJECT_ID)
sub = df[df['id'] == SUBJECT_ID].copy().reset_index(drop=True)

# ✅ Drop rows missing CGM
sub = sub.dropna(subset=['CGM']).reset_index(drop=True)

# ✅ Auxiliary features
aux_features = [
    'carbs','bolus','basal','galvanic_skin_response','skin_temp','acceleration',
    'workout_intensity','workout_duration','heartrate','air_temp','steps'
]
aux_features = [c for c in aux_features if c in sub.columns]
print("Using AUX:", aux_features)

sub[aux_features] = sub[aux_features].fillna(0.0)
sub['CGM'] = sub['CGM'].astype(float)

# ✅ Build sliding windows
def build_windows(data, lb, ph):
    X_cgm, X_aux, Y, times = [], [], [], []
    for i in range(len(data) - (lb + ph) + 1):
        seq = data.iloc[i:i + lb]
        future = data.iloc[i + lb:i + lb + ph]
        if future['CGM'].isnull().any():
            continue
        X_cgm.append(seq['CGM'].values.reshape(-1,1))
        X_aux.append(seq[aux_features].values)
        Y.append(future['CGM'].values)
        times.append(data.iloc[i + lb]['date'])
    return np.array(X_cgm), np.array(X_aux), np.array(Y), np.array(times)

X_cgm, X_aux, Y, T = build_windows(sub, LOOKBACK_STEPS, N_PRED_STEPS)
print("Windows:", X_cgm.shape)

# ✅ Train/test split
split = int(len(X_cgm)*(1-TEST_FRACTION))
X_cgm_tr, X_cgm_te = X_cgm[:split], X_cgm[split:]
X_aux_tr, X_aux_te = X_aux[:split], X_aux[split:]
Y_tr, Y_te = Y[:split], Y[split:]
T_te = T[split:]

print("Train:", len(X_cgm_tr), "Test:", len(X_cgm_te))

# ✅ Scaling
sc_cgm = StandardScaler().fit(X_cgm_tr.reshape(-1,1))
sc_aux = StandardScaler().fit(X_aux_tr.reshape(-1, X_aux_tr.shape[2]))
sc_y   = StandardScaler().fit(Y_tr.reshape(-1,1))

def scale(Xcgm, Xaux, Yy=None):
    Xcs = sc_cgm.transform(Xcgm.reshape(-1,1)).reshape(Xcgm.shape)
    Xas = sc_aux.transform(Xaux.reshape(-1, Xaux.shape[2])).reshape(Xaux.shape)
    if Yy is not None:
        Ys = sc_y.transform(Yy.reshape(-1,1)).reshape(Yy.shape)
        return Xcs, Xas, Ys
    return Xcs, Xas

Xc_tr_s, Xa_tr_s, Y_tr_s = scale(X_cgm_tr, X_aux_tr, Y_tr)
Xc_te_s, Xa_te_s, Y_te_s = scale(X_cgm_te, X_aux_te, Y_te)

# ✅ Build dual-input LSTM
input_cgm = Input(shape=(LOOKBACK_STEPS,1))
x1 = Masking()(input_cgm)
x1 = LSTM(LSTM_UNITS,return_sequences=True)(x1)
x1 = Dropout(DROPOUT)(x1)
x1 = LSTM(LSTM_UNITS//2)(x1)

input_aux = Input(shape=(LOOKBACK_STEPS,X_aux.shape[2]))
x2 = Masking()(input_aux)
x2 = LSTM(LSTM_UNITS,return_sequences=True)(x2)
x2 = Dropout(DROPOUT)(x2)
x2 = LSTM(LSTM_UNITS//2)(x2)

x = concatenate([x1,x2])
x = Dense(128,activation='relu')(x)
x = Dropout(DROPOUT)(x)
output = Dense(N_PRED_STEPS)(x)

model = Model(inputs=[input_cgm,input_aux], outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
model.summary()

# ✅ Train model
val_split = int(0.9 * len(Xc_tr_s))
history = model.fit(
    [Xc_tr_s[:val_split], Xa_tr_s[:val_split]], Y_tr_s[:val_split],
    validation_data=([Xc_tr_s[val_split:], Xa_tr_s[val_split:]], Y_tr_s[val_split:]),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[EarlyStopping(patience=6,restore_best_weights=True),
               ReduceLROnPlateau(patience=4)],
    verbose=1
)

# ✅ Predict + inverse scale
Y_pred_s = model.predict([Xc_te_s, Xa_te_s], batch_size=BATCH_SIZE)
Y_pred = sc_y.inverse_transform(Y_pred_s.reshape(-1,1)).reshape(Y_pred_s.shape)
Y_true = Y_te

# ✅ Metrics
print("--- METRICS ---")
print("Flattened MSE:", mean_squared_error(Y_true.ravel(), Y_pred.ravel()))
print("Flattened MAE:", mean_absolute_error(Y_true.ravel(), Y_pred.ravel()))

# ✅ Per-step metrics
print("\nStep-wise metrics:")
for i in range(N_PRED_STEPS):
    mse_i = mean_squared_error(Y_true[:,i], Y_pred[:,i])
    mae_i = mean_absolute_error(Y_true[:,i], Y_pred[:,i])
    print(f" +{STEP_MIN*(i+1)} min -> MSE: {mse_i:.2f} | MAE: {mae_i:.2f}")

# ✅ Plot a single sample (mid test)
idx = len(Xc_te_s)//2
anchor_time = T_te[idx]
future_times = [anchor_time + pd.Timedelta(minutes=STEP_MIN*(i+1))
                for i in range(N_PRED_STEPS)]

# Zero-order hold baseline
last_val = X_cgm_te[idx][-1,0]
y_zero = np.repeat(last_val, N_PRED_STEPS)

plt.figure(figsize=(10,4))
plt.plot(future_times, Y_true[idx], 'o-', label='Actual')
plt.plot(future_times, Y_pred[idx], 'o--', label='LSTM')
plt.plot(future_times, y_zero,  'o:', label=f'ZOH baseline ({last_val:.1f})')
plt.grid(); plt.legend()
plt.title(f'Multi-Step CGM Prediction — {SUBJECT_ID} | Anchor: {anchor_time}')
plt.xlabel("Time"); plt.ylabel("CGM (mg/dL)")
plt.gcf().autofmt_xdate()
plt.show()

# ✅ Quick curve for first-step predictions over whole test set
plt.figure(figsize=(12,4))
plt.plot(T_te, Y_true[:,0], '.-', label='Actual +5min')
plt.plot(T_te, Y_pred[:,0], '.--', label='LSTM +5min')
plt.grid(); plt.legend()
plt.title('First-Step Predictions Across Test Timeline')
plt.ylabel("CGM"); plt.xlabel("Time")
plt.gcf().autofmt_xdate()
plt.show()


In [ ]:
# --- Global-model training + per-subject evaluation/plots (paste into a notebook) ---
# - Train a single LSTM model on combined training data from subjects_to_plot subjects
# - Evaluate per subject and plot Actual vs Predicted CGM at t+30min
# - X-axis is minutes (0..30)
# Requirements: put "../data/OhioT1DM.csv" next to your notebook.

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt

# ----------------- parameters -----------------
DATA_PATH = "../data/OhioT1DM.csv"
n_in = 24           # input window in steps (12 * 5min = 60 min history)
n_out = 1           # output horizon in steps (6 * 5min = 30 min ahead)
epochs = 50
batch_size = 16
subjects_to_plot = 3
# ------------------------------------------------

# helper: sliding windows
def make_sequences(values, n_in, n_out):
    X, y = [], []
    for i in range(len(values) - n_in - n_out + 1):
        X.append(values[i:i+n_in])
        # y is the CGM column (first column)
        y.append(values[i+n_in:i+n_in+n_out, 0])
    return np.array(X), np.array(y)

# read data
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
feature_cols = ["CGM","carbs","bolus","basal","galvanic_skin_response","skin_temp",
                "acceleration","workout_intensity","workout_duration","heartrate","air_temp","steps"]

ids = df['id'].unique()[:subjects_to_plot]

# --- first pass: collect raw per-subject arrays so we can fit global scalers ---
raw_per_subject = {}
for sid in ids:
    sub = df[df['id']==sid].sort_values('date').copy()
    sub = sub.set_index('date')
    # reindex to consistent 5-min sampling
    full_idx = pd.date_range(sub.index.min(), sub.index.max(), freq='5T')
    sub = sub.reindex(full_idx)
    # sensible fills
    sub['CGM'] = sub['CGM'].ffill().bfill()
    sub[['carbs','bolus','basal','workout_intensity','workout_duration','steps']] = \
        sub[['carbs','bolus','basal','workout_intensity','workout_duration','steps']].fillna(0)
    sub[['galvanic_skin_response','skin_temp','acceleration','heartrate','air_temp']] = \
        sub[['galvanic_skin_response','skin_temp','acceleration','heartrate','air_temp']].ffill().bfill().fillna(0)

    data = sub[feature_cols].values.astype(float)   # shape (T, n_features)
    raw_per_subject[sid] = {'data': data, 'full_idx': full_idx}

# check we have data
if len(raw_per_subject) == 0:
    raise RuntimeError("No subject data found for the chosen ids.")

# concatenate across subjects to fit global scalers (CGM scaler + "others" scaler)
all_cg = np.vstack([raw_per_subject[s]['data'][:, 0:1] for s in raw_per_subject])
all_others = np.vstack([raw_per_subject[s]['data'][:, 1:] for s in raw_per_subject])

cg_scaler = MinMaxScaler().fit(all_cg)
others_scaler = MinMaxScaler().fit(all_others)

# --- second pass: create sequences, split per-subject (chronological), and collect global training set ---
X_train_list = []
y_train_list = []
per_subject_test = {}   # store X_test, y_test, times_test for each subject

for sid, info in raw_per_subject.items():
    data = info['data']
    full_idx = info['full_idx']

    # apply global scalers
    cg_scaled = cg_scaler.transform(data[:, 0:1])
    others_scaled = others_scaler.transform(data[:, 1:])
    data_scaled = np.hstack([cg_scaled, others_scaled])

    # make sequences
    X, y = make_sequences(data_scaled, n_in, n_out)
    if len(X) == 0:
        print(f"Warning: subject {sid} has insufficient data for windows -> skipping.")
        continue

    # timestamps correspond to the final predicted step (t + (n_out-1)*5min)
    timestamps = full_idx[n_in : n_in + len(y)] + pd.to_timedelta(5*(n_out-1), unit='m')

    # chronological split for each subject: first 70% -> train, last 30% -> test
    split = int(0.7 * len(X))
    if split < 1:
        # too small to get training portion
        print(f"Warning: subject {sid} has too few windows for training/test split -> skipping.")
        continue

    X_train_list.append(X[:split])
    y_train_list.append(y[:split])

    per_subject_test[sid] = {
        'X_test': X[split:],
        'y_test': y[split:],
        'times_test': timestamps[split:]
    }

# build global training arrays
if len(X_train_list) == 0:
    raise RuntimeError("No training data collected across subjects (check subject selection and data).")

X_train_all = np.vstack(X_train_list)
y_train_all = np.vstack(y_train_list)

print(f"Global training samples: {len(X_train_all)}   (n_in={n_in}, n_out={n_out})")

# --- build single global model and train once ---
n_features = X_train_all.shape[2]

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(n_in, n_features)),
    LSTM(64, return_sequences=False),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(n_out)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# shuffle the global training set (we combined chronological train parts from each subject)
history = model.fit(
    X_train_all, y_train_all,
    epochs=epochs,
    batch_size=batch_size,
    verbose=1,
    shuffle=True,
    validation_split=0.1,
    callbacks=callbacks
)

# --- per-subject evaluation and plotting ---
def inv_cgm_with_cg_scaler(scaled_vals):
    # scaled_vals shape: (n_samples, n_out)
    out = []
    for step_idx in range(scaled_vals.shape[1]):
        col = scaled_vals[:, step_idx].reshape(-1, 1)
        inv = cg_scaler.inverse_transform(col)[:, 0]
        out.append(inv)
    return np.stack(out, axis=1)  # shape (n_samples, n_out)

plt.figure(figsize=(10, 4 * len(per_subject_test)))
for idx_i, sid in enumerate(per_subject_test.keys()):
    info = per_subject_test[sid]
    X_test = info['X_test']
    y_test = info['y_test']
    times_test = info['times_test']

    if len(X_test) == 0:
        print(f"No test samples for subject {sid}, skipping plot.")
        continue

    y_pred = model.predict(X_test, batch_size=batch_size)

    # invert CGM scaling cleanly
    y_test_inv = inv_cgm_with_cg_scaler(y_test)
    y_pred_inv = inv_cgm_with_cg_scaler(y_pred)

    # take the final forecast step (t + 30 min)
    actual_30 = y_test_inv[:, -1]
    pred_30 = y_pred_inv[:, -1]

    # minutes since start of this subject's test portion
    t0 = times_test[0]
    minutes = np.array([(t - t0).total_seconds() / 60.0 for t in times_test])

    # only show up to 30 minutes (mask)
    mask = minutes <= 30.0
    if mask.sum() == 0:
        # fallback: plot up to first min(30, len(points)) points (use their minutes)
        idx_plot = np.arange(min(30, len(minutes)))
        minutes_plot = minutes[idx_plot]
        actual_plot = actual_30[idx_plot]
        pred_plot = pred_30[idx_plot]
    else:
        minutes_plot = minutes[mask]
        actual_plot = actual_30[mask]
        pred_plot = pred_30[mask]

    ax = plt.subplot(len(per_subject_test), 1, idx_i+1)
    ax.plot(minutes_plot, actual_plot, label='Actual CGM (t+30min)', linewidth=1)
    ax.plot(minutes_plot, pred_plot, label='Predicted CGM (t+30min)', linewidth=1)
    ax.set_ylabel('BG (mg/dL)')
    ax.set_xlabel('Minutes since start of test segment')
    ax.set_title(f'Subject id={sid}   (history {n_in*5} min -> predict {n_out*5} min ahead)')
    ax.set_xlim(0, 30)   # enforce 0..30 minutes
    ax.legend()
    ax.grid(True)

plt.tight_layout()
# ---------------------------------------------------------


## Test ML corrector:

In [ ]:
# Cell 5 (REPLACEMENT): Hybrid ML-corrector (stable training)
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# ---------- Build training set where target = residual (true - bergman_pred) ----------
X_corr = []
y_corr = []
# Also keep an index list if you want to debug samples
sample_meta = []

for sid, info in per_subject_train.items():
    Xs = info['X']   # scaled inputs
    if Xs.shape[0] == 0:
        continue
    subj = per_subject[sid]
    L = subj.shape[0]
    # loop windows aligned with create_sequences
    for i in range(N_IN, L - N_OUT + 1):
        xin = Xs[i - N_IN]  # input sequence aligned with create_sequences indexing
        # raw seq needed for bergman baseline
        G_init = subj['CGM_smoothed'].values[i-1]
        bolus_seq = subj['bolus'].values[i:i+N_OUT]
        carbs_seq = subj['carbs'].values[i:i+N_OUT]
        berg_pred = simulate_bergman(bergman_model['params'], (G_init, bolus_seq, carbs_seq, None))
        berg_pred_scaled = cg_scaler.transform(berg_pred.reshape(-1,1)).reshape(-1,1)
        y_true_scaled = cg_scaler.transform(subj['CGM_smoothed'].values[i:i+N_OUT].reshape(-1,1)).reshape(-1,1)
        resid = (y_true_scaled - berg_pred_scaled).reshape(N_OUT)  # flattened residual (scaled units)
        X_corr.append(xin.astype(np.float32))
        y_corr.append(resid.astype(np.float32))
        sample_meta.append((sid, i))

if len(X_corr) == 0:
    raise RuntimeError("No training data found for ML-corrector. Check preprocessing.")

X_corr = np.stack(X_corr)   # (N_samples, n_in, n_feats)
y_corr = np.stack(y_corr)   # (N_samples, n_out)

# ---------- Sanitize: remove NaN/Inf samples ----------
def finite_mask(X, y):
    # X: (N, n_in, n_feats); y: (N, n_out)
    mask = np.isfinite(y).all(axis=1) & np.isfinite(X).all(axis=(1,2))
    return mask

mask = finite_mask(X_corr, y_corr)
n_before = X_corr.shape[0]
if not mask.all():
    X_corr = X_corr[mask]
    y_corr = y_corr[mask]
    # also shrink sample_meta if needed
    dropped = n_before - X_corr.shape[0]
    print(f"[ML-corrector] Dropped {dropped} samples with NaN/Inf; {X_corr.shape[0]} samples remain.")

# cast to float32 (TF-friendly)
X_corr = X_corr.astype(np.float32)
y_corr = y_corr.astype(np.float32)

n_samples = X_corr.shape[0]
print("[ML-corrector] Training samples:", n_samples)

# ---------- Check shapes ----------
n_in = X_corr.shape[1]
n_feats = X_corr.shape[2]
n_out = y_corr.shape[1]

# ---------- Stable loss: MSE + small L1 on residuals (works for negative residuals) ----------
def mse_l1_residual_loss(y_true, y_pred):
    # both are residuals in scaled units; safe for negatives
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    l1  = tf.reduce_mean(tf.abs(y_true - y_pred))
    return mse + 0.01 * l1

# ---------- Model ----------
tf.keras.backend.clear_session()
model_corr = Sequential([
    LSTM(32, return_sequences=False, input_shape=(n_in, n_feats)),
    Dense(n_out, activation='linear')
])

model_corr.compile(optimizer=Adam(1e-3), loss=mse_l1_residual_loss, metrics=['mse'])
print(model_corr.summary())

# ---------- Fit ----------
# Use a small validation split from the training windows (subject-split already ensured globally)
history = model_corr.fit(X_corr, y_corr, epochs=12, batch_size=64, validation_split=0.05, verbose=2)

# ---------- Test-time predictor ----------
def ml_corrector_predict_subject(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0]==0:
        return np.zeros((0, N_OUT, 1))
    subj = per_subject[sid]
    L = subj.shape[0]
    preds_all = []
    # Batch predict residuals for efficiency
    Xs = info['X']
    if Xs.shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    resid_preds = model_corr.predict(Xs, batch_size=128)  # shape (n_windows, n_out)
    # Now compute bergman baseline for each window (vectorized)
    n_windows = Xs.shape[0]
    G_inits = subj['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    bolus_mat = np.zeros((n_windows, N_OUT))
    carbs_mat = np.zeros((n_windows, N_OUT))
    for w in range(n_windows):
        start = N_IN + w
        bolus_mat[w, :] = subj['bolus'].values[start:start+N_OUT]
        carbs_mat[w, :] = subj['carbs'].values[start:start+N_OUT]
    p1, p2, p3, k_i, Gb = bergman_model['params']
    berg_preds = np.zeros((n_windows, N_OUT))
    G = G_inits.copy().astype(float)
    I = np.zeros(n_windows)
    for t in range(N_OUT):
        I = I * math.exp(-k_i) + bolus_mat[:, t]
        meal_effect = p3 * carbs_mat[:, t]
        dG = -p1 * (G - Gb) - p2 * I + meal_effect
        G = G + dG * 1.0
        berg_preds[:, t] = G
    berg_preds_scaled = cg_scaler.transform(berg_preds.reshape(-1,1)).reshape(-1, N_OUT)
    corrected_scaled = berg_preds_scaled + resid_preds
    return corrected_scaled.reshape(n_windows, N_OUT, 1)

ml_corrector_model = {"lstm_corrector": model_corr, "predict": ml_corrector_predict_subject}


In [ ]:
# Cell 5: Hybrid ML-corrector (LSTM corrector on residuals)
# Processing data, Feature engineering & selection, Training model, Test model

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# ---------- Processing: build training set where target = residual (true - bergman_pred) ----------
X_corr = []
y_corr = []
for sid, info in per_subject_train.items():
    Xs = info['X']   # scaled inputs
    if Xs.shape[0] == 0:
        continue
    # compute bergman predictions for this subject's windows using bergman_model
    # we'll get scaled bergman preds per-sample by re-running bergman on internal data alignment
    subj = per_subject[sid]
    L = subj.shape[0]
    for i in range(N_IN, L - N_OUT + 1):
        xin = Xs[i - N_IN]  # input sequence aligned with create_sequences indexing
        # raw seq needed
        G_init = subj['CGM_smoothed'].values[i-1]
        bolus_seq = subj['bolus'].values[i:i+N_OUT]
        carbs_seq = subj['carbs'].values[i:i+N_OUT]
        berg_pred = simulate_bergman(bergman_model['params'], (G_init, bolus_seq, carbs_seq, None))
        berg_pred_scaled = cg_scaler.transform(berg_pred.reshape(-1,1)).reshape(-1,1)
        y_true_scaled = cg_scaler.transform(subj['CGM_smoothed'].values[i:i+N_OUT].reshape(-1,1)).reshape(-1,1)
        resid = (y_true_scaled - berg_pred_scaled).reshape(N_OUT)  # flatten n_out
        X_corr.append(xin)
        y_corr.append(resid)
if len(X_corr)==0:
    raise RuntimeError("No training data found for ML-corrector.")
X_corr = np.stack(X_corr)  # (N_samples, n_in, n_feats)
y_corr = np.stack(y_corr)  # (N_samples, n_out)

# ---------- Feature engineering: keep as-is ----------
n_in = X_corr.shape[1]
n_feats = X_corr.shape[2]
n_out = y_corr.shape[1]

# ---------- Surrogate loss same as LSTM (we use RMSE + soft glycemia) ----------
# re-use lstm_surrogate_loss defined earlier by adapting for shape (batch, n_out)
# We'll re-create a simple callable loss for compile (since earlier was bound to previous graph)
import tensorflow as tf
alpha = 0.6
k_soft = 0.08
min_v = float(cg_scaler.data_min_[0]); max_v = float(cg_scaler.data_max_[0])

def soft_glycemia_probs_flat(x_flat):
    # x_flat is (batch, n_out) scaled in [0,1]
    x = tf.reshape(x_flat, (-1,))
    x_mgdl = x * (max_v - min_v) + min_v
    hypo = tf.sigmoid(k_soft * (70.0 - x_mgdl))
    hyper = tf.sigmoid(k_soft * (x_mgdl - 180.0))
    norm = 1.0 - tf.clip_by_value(hypo + hyper, 0.0, 1.0)
    probs = tf.stack([hypo, norm, hyper], axis=1)
    return probs

def corrector_loss(y_true, y_pred):
    # y_true/y_pred shape (batch, n_out)
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    rmse = tf.sqrt(mse + 1e-8)
    # convert residual + bergman_pred to full glucose for class probs
    # We cannot access bergman_pred here; instead we compute soft penalty on the corrected final glucose by
    # assuming that later in the pipeline corrected_pred_full = bergman_pred_scaled + y_pred.
    # During training we can approximate the full by using bergman estimates computed offline:
    # For simplicity we apply soft class penalty only on y_true (resid->y_true+bergman) via a proxy:
    soft_diff = 0.0  # keep simple; main objective is RMSE of residuals
    return rmse + 0.1 * soft_diff


# paper constants
C = 32.9170
w_left = 19.0
w_right = 1.0
T_mgdl = 105.0  # target BG in mg/dL

# slope-cost constants (tuneable)
a = 1.0
b = 1.0

eps = 1e-8

# helper: convert scaled values back to mg/dL using your scaler
min_v = float(cg_scaler.data_min_[0])
max_v = float(cg_scaler.data_max_[0])

def _to_mgdl(tensor_scaled):
    """tensor_scaled: shape (batch, n_out) or (...). Inverse min-max scaling to mg/dL."""
    return tensor_scaled * (max_v - min_v) + min_v

def _zone_cost(bg_mgdl):
    """bg_mgdl: tensor of BG in mg/dL, any shape. Returns zone cost same shape."""
    # log difference squared
    # add eps to log to avoid -inf/nan
    log_bg = tf.math.log(bg_mgdl + eps)
    log_T = tf.math.log(T_mgdl + eps)
    diff2 = tf.square(log_bg - log_T)

    # piecewise multiply by C*w_left or C*w_right
    left_mask = tf.cast(bg_mgdl < T_mgdl, dtype=bg_mgdl.dtype)
    right_mask = 1.0 - left_mask
    cost = C * (w_left * diff2 * left_mask + w_right * diff2 * right_mask)
    return cost

def _slope_cost(delta_bg_mgdl):
    """delta_bg_mgdl: difference in mg/dL. Uses formula in paper but converted to mmol/L inside."""
    # convert mg/dL -> mmol/L by /18.0
    delta_mmol = delta_bg_mgdl / 18.0
    # piecewise: if delta < 0 use 2b*(Δ)^2, else a*(Δ)^2
    neg_mask = tf.cast(delta_mmol < 0.0, dtype=delta_mmol.dtype)
    pos_mask = 1.0 - neg_mask
    cost = (2.0 * b) * tf.square(delta_mmol) * neg_mask + a * tf.square(delta_mmol) * pos_mask
    return cost

def lstm_paper_loss_hybrid(y_true, y_pred):
    """
    y_true, y_pred: expected shapes (batch, n_out) because you flatten targets before fit.
    Returns scalar loss: mean( weight * squared_error ) + alpha * mean( abs_error )
    where weight = zone_cost(y_true_mgdl) + slope_cost(delta_y_true_mgdl) + 1
    """
    # ensure float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # convert to mg/dL
    y_true_mgdl = _to_mgdl(y_true)
    y_pred_mgdl = _to_mgdl(y_pred)

    # compute slope/delta along horizon axis (axis=1). For first horizon delta=0
    # shape (batch, n_out)
    y_true_shifted = tf.concat([y_true_mgdl[:, :1] * 0.0, y_true_mgdl[:, :-1]], axis=1)
    delta_true = y_true_mgdl - y_true_shifted

    # zone cost and slope cost per element
    zone = _zone_cost(y_true_mgdl)          # shape (batch,n_out)
    slope = _slope_cost(delta_true)        # shape (batch,n_out)
    weights = zone + slope + 1.0           # baseline +1 as in paper

    # weighted MSE (average over all elements)
    se = tf.square(y_pred - y_true)        # in scaled units
    weighted_se = weights * se
    mse_weighted = tf.reduce_mean(weighted_se)

    # L1 term (in scaled units)
    l1 = tf.reduce_mean(tf.abs(y_pred - y_true))

    # Paper returns MSE_weighted + alpha * L1
    loss = mse_weighted + alpha * l1

    return loss

# ---------- Training model ----------
from tensorflow.keras import backend as K
tf.keras.backend.clear_session()
model_corr = Sequential([
    LSTM(32, return_sequences=False, input_shape=(n_in, n_feats)),
    Dense(n_out, activation='linear')
])
model_corr.compile(optimizer=Adam(1e-3), loss=lstm_paper_loss_hybrid)
print(model_corr.summary())
model_corr.fit(X_corr, y_corr, epochs=12, batch_size=64, verbose=2)

# ---------- Test model: produce corrected predictions per subject ----------
def ml_corrector_predict_subject(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0]==0:
        return np.zeros((0, N_OUT, 1))
    subj = per_subject[sid]
    L = subj.shape[0]
    preds_all = []
    for i in range(N_IN, L - N_OUT + 1):
        xin = info['X'][i - N_IN]  # aligned input
        xin_batch = xin.reshape(1, n_in, n_feats)
        resid_pred = model_corr.predict(xin_batch)[0]  # shape (n_out,)
        # bergman pred for this window
        G_init = subj['CGM_smoothed'].values[i-1]
        bolus_seq = subj['bolus'].values[i:i+N_OUT]
        carbs_seq = subj['carbs'].values[i:i+N_OUT]
        berg_pred = simulate_bergman(bergman_model['params'], (G_init, bolus_seq, carbs_seq, None))
        berg_pred_scaled = cg_scaler.transform(berg_pred.reshape(-1,1)).reshape(-1)
        corrected = berg_pred_scaled + resid_pred
        preds_all.append(corrected.reshape(-1,1))
    if len(preds_all) == 0:
        return np.zeros((0,N_OUT,1))
    return np.stack(preds_all)

ml_corrector_model_new = {"lstm_corrector": model_corr, "predict": ml_corrector_predict_subject}
